# 03. Reducing variance

Randomizing makes the estimate **unbiased**, but not necessarily **precise**. If you measure covariates correlated with the outcome, you can use them to shrink the standard error without introducing bias. Two routes in the library: `LinEstimator` (regression adjustment) and `CUPED` (using a pre-experiment period).

## The experiment

The outcome is strongly explained by a covariate `x` and a pre-experiment measure `y_pre`. The true effect is `0.5`.

In [ ]:
import numpy as np
import pandas as pd

from skxperiments.core.assignment import CRDAssignment
from skxperiments.design.crd import CRD

rng = np.random.default_rng(11)
n = 200
x = rng.normal(size=n)
y_pre = rng.normal(size=n)
base = 1.5 * x + 1.5 * y_pre + rng.normal(size=n) * 0.5   # baseline explained by x and y_pre
tau = 0.5
y0 = base
y1 = base + tau

df = pd.DataFrame({"x": x, "y_pre": y_pre})
design = CRD(p=0.5, seed=11)
assignment = design.randomize(df)
t = assignment.data_[assignment.treatment_col_].to_numpy()
data = assignment.data_.copy()
data["y"] = np.where(t == 1, y1, y0)
assignment = CRDAssignment(
    data=data, treatment_col=assignment.treatment_col_, design=design, seed=11
)

## Same estimate, different precision

The three estimators target the same ATE. We use `BootstrapCI` (which accepts any scalar estimator) to compare the **standard error**. Lin and CUPED should have a smaller SE than the plain difference in means.

In [ ]:
from skxperiments.estimators.cuped import CUPED
from skxperiments.estimators.difference_in_means import DifferenceInMeans
from skxperiments.estimators.lin_estimator import LinEstimator
from skxperiments.inference import BootstrapCI

estimators = {
    "DifferenceInMeans": DifferenceInMeans(outcome_col="y"),
    "Lin (x)": LinEstimator(outcome_col="y", covariates=["x"]),
    "CUPED (y_pre)": CUPED(outcome_col="y", pre_experiment_col="y_pre"),
}
rows = []
for name, est in estimators.items():
    res = BootstrapCI(
        estimator=est, method="percentile", n_resamples=800, seed=0
    ).fit(assignment).estimate()
    rows.append({"estimator": name, "ATE": res.ate, "SE": res.se})
pd.DataFrame(rows)

## What we learned

- Covariates correlated with the outcome **reduce the variance** of the estimate while keeping it unbiased.
- `LinEstimator` adjusts for covariates measured in the experiment; `CUPED` uses a pre-experiment measure.
- A smaller SE means a narrower CI and more power at the same sample size.

**Next:** `04. Balance and rerandomization` shows how to check and improve covariate balance.